# Credit Card Graph Anomaly Detection

**Portfolio edition | Oklahoma purchase-card transactions | July 2013–June 2014**

This notebook reconstructs the analytical pipeline from the uploaded academic assignment, using a relative data path and omitting saved outputs that could expose cardholder or vendor identifiers. It is not a fraud classifier. The original assignment reported 442,458 transactions, 39 flagged node-month observations, and 466 large adjacent-month percentile movements. Run this notebook to verify results in your environment.

## 1. Load and clean transactions

Place `purchase_credit_card.csv.zip` (containing `purchase_credit_card.csv`) in `data/`. Do not commit transaction-level data to this public repository. The dataset contains potentially identifying information.

In [ ]:
from pathlib import Path
import zipfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import sparse
from scipy.stats import linregress

archive = Path("data/purchase_credit_card.csv.zip")
if not archive.exists():
    raise FileNotFoundError("Place purchase_credit_card.csv.zip in data/")
with zipfile.ZipFile(archive) as z:
    df = pd.read_csv(z.open("purchase_credit_card.csv"), low_memory=False)
print("Original shape:", df.shape)
df = df.dropna(subset=["Cardholder First Initial", "Cardholder Last Name", "Vendor"]).copy()
for col in ["Cardholder First Initial", "Cardholder Last Name", "Vendor"]:
    df[col] = df[col].astype(str).str.strip().str.upper()
df["Cardholder"] = (df["Agency Number"].astype(str) + " | " +
                    df["Cardholder First Initial"] + " " + df["Cardholder Last Name"])
df["Vendor"] = df["Vendor"].str.replace(r"\s+", " ", regex=True)
df = df[(df["Cardholder"].str.len() > 5) & (df["Vendor"] != "")]
print("Usable transactions:", len(df))
print("Months:", df["Year-Month"].nunique())

## 2. Construct monthly bipartite networks and compute PageRank

Cardholders and vendors are separate node types. Repeated transactions increase edge weights. The graph is undirected, and PageRank uses damping factor 0.85. Residual anomaly screening compares log PageRank against log degree among nodes with degree ≥ 7, separately by node type. The threshold is |z| > 4.

In [ ]:
def monthly(group, month):
    edges = group.groupby(["Cardholder", "Vendor"], sort=False).agg(
        txns=("Amount", "size")
    ).reset_index()
    holders, hi = np.unique(edges["Cardholder"], return_inverse=True)
    vendors, vi = np.unique(edges["Vendor"], return_inverse=True)
    nh, nv = len(holders), len(vendors)
    n = nh + nv
    bipartite = sparse.coo_matrix(
        (edges["txns"].to_numpy(dtype=float), (hi, vi)), shape=(nh, nv)
    ).tocsr()
    adjacency = sparse.bmat([[None, bipartite], [bipartite.T, None]], format="csr")
    strength = np.asarray(adjacency.sum(axis=1)).ravel()
    transition = sparse.diags(1 / strength) @ adjacency
    rank = np.full(n, 1 / n)
    alpha = 0.85
    for _ in range(300):
        next_rank = (1 - alpha) / n + alpha * (transition.T @ rank)
        if np.abs(next_rank - rank).sum() < 1e-10:
            rank = next_rank
            break
        rank = next_rank
    degree = np.asarray((adjacency > 0).sum(axis=1)).ravel()
    rows = pd.DataFrame({
        "month": month,
        "type": ["Cardholder"] * nh + ["Vendor"] * nv,
        "name": list(holders) + list(vendors),
        "rank": rank, "degree": degree, "txns": strength.astype(int)
    })
    rows["rank_pct"] = rows.groupby("type")["rank"].rank(pct=True) * 100
    rows["resid_z"] = np.nan
    for node_type in ["Cardholder", "Vendor"]:
        mask = (rows["type"] == node_type) & (rows["degree"] >= 7)
        if mask.sum() < 3:
            continue
        x = np.log(rows.loc[mask, "degree"])
        y = np.log(rows.loc[mask, "rank"])
        slope, intercept, *_ = linregress(x, y)
        residual = y - (intercept + slope * x)
        sd = residual.std(ddof=1)
        if sd > 0:
            rows.loc[mask, "resid_z"] = (residual - residual.mean()) / sd
    rows["anomaly"] = rows["resid_z"].abs() > 4
    return rows

monthly_scores = pd.concat(
    [monthly(group, int(month)) for month, group in df.groupby("Year-Month")],
    ignore_index=True
)
flagged = monthly_scores.loc[monthly_scores["anomaly"]].copy()
summary = monthly_scores.groupby(["month", "type"]).agg(
    nodes=("rank", "size"), anomalies=("anomaly", "sum"),
    median_degree=("degree", "median")
).reset_index()
print(summary.to_string(index=False))
print("Flagged node-months:", len(flagged))

## 3. Visualize flagged counts without exposing identities

The plot uses only aggregate monthly counts. This is appropriate for a public portfolio screenshot.

In [ ]:
pivot = summary.pivot(index="month", columns="type", values="anomalies").fillna(0)
ax = pivot.plot(marker="o", figsize=(10, 4))
ax.set(title="PageRank residual anomalies by month", xlabel="Month",
       ylabel="Flagged node-months")
ax.grid(alpha=0.2)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Detect large consecutive-month changes

Percentile ranks are compared within node type and month. The screen requires adjacent calendar months, degree ≥ 7 and an absolute change of at least 50 percentile points.

In [ ]:
timeline = monthly_scores.sort_values(["type", "name", "month"]).copy()
timeline["previous_pct"] = timeline.groupby(["type", "name"])["rank_pct"].shift()
timeline["previous_month"] = timeline.groupby(["type", "name"])["month"].shift()
timeline["change_pct_points"] = timeline["rank_pct"] - timeline["previous_pct"]
current_month = pd.to_datetime(timeline["month"].astype(str), format="%Y%m")
prior_month = pd.to_datetime(
    timeline["previous_month"].astype("Int64").astype(str),
    format="%Y%m", errors="coerce"
)
timeline["adjacent"] = (
    current_month.dt.to_period("M") - prior_month.dt.to_period("M")
).apply(lambda x: x.n if pd.notna(x) else 0) == 1
changes = timeline.loc[
    timeline["adjacent"] &
    (timeline["degree"] >= 7) &
    (timeline["change_pct_points"].abs() >= 50)
].copy()
print("Large adjacent-month movements:", len(changes))
print("Unique entities with movements:", changes[["type", "name"]].drop_duplicates().shape[0])

## 5. Compare with transaction-spending signals

This is a conceptual cross-check: a structurally unusual cardholder may or may not also be a top spender. No ground-truth fraud labels are available.

In [ ]:
spending = df.groupby(["Year-Month", "Cardholder"]).agg(
    total_spend=("Amount", "sum"),
    transaction_count=("Amount", "size")
).reset_index().rename(columns={"Year-Month": "month", "Cardholder": "name"})
spending["spend_pct"] = spending.groupby("month")["total_spend"].rank(pct=True) * 100
holder_flags = flagged.loc[flagged["type"] == "Cardholder"].merge(
    spending, on=["month", "name"], how="left"
)
print("Flagged cardholder-months:", len(holder_flags))
print("Also in top 1% of monthly spending:",
      int((holder_flags["spend_pct"] >= 99).sum()))

## 6. Conclusions and limitations

Monthly PageRank can surface changes in network importance that transaction amount alone may miss. The original assignment's reported counts should be verified by executing this version against the same data. PageRank scores depend on network construction, period, and node coverage. Screening thresholds are exploratory. **An anomaly is not confirmed fraud**. Further evaluation would require external validation labels, sensitivity analysis, and review of legitimate procurement patterns.

**Privacy:** This notebook intentionally avoids printing or saving cardholder/vendor-level flagged tables in its public version.